In [ ]:
import torch
import numpy as np
from typing import List,Literal,Type 

In [ ]:
import sys
from pathlib import Path

SRC_DIR = Path.cwd().resolve()

# Si le kernel a été lancé depuis la racine du projet
if SRC_DIR.name != "src":
    SRC_DIR = SRC_DIR / "src"

sys.path.insert(0, str(SRC_DIR))

In [ ]:
from evaluation.scoring import mean_residual_score, ScoreMethod
from purification.purifier import Purifier
from purification.config import PurificationConfig
from purification.step_curves import (
    linear_timesteps,
    quadratic_timesteps,
    uniform_log_snr_steps,
)

In [ ]:
from inference import load_diffusion_model, ddim_reverse_step,reverse_diffusion_ddim, make_ddim_timesteps
from diffusion import DiffusionSchedule,diffuse
from network import UNetDiffusion

In [ ]:
N_STEPS = 1000
T_START = 600
K = 4
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
device

In [ ]:
model,schedule = load_diffusion_model(ckpt_path="../models/unet32.pt",device=device)


In [ ]:
patch = np.load("../dataset/test/patch_9107_22.npy").astype(np.float32).squeeze()
patch = (patch - patch.mean()) / (patch.std() + 1e-8)

x0 = torch.from_numpy(patch)[None, None].to(schedule.device)  # (1, 1, 48, 48)
t = torch.tensor([T_START], device=x0.device)

xt, noise = diffuse(x0, t, schedule)

In [ ]:
purf = Purifier(model,schedule)
purf.reconstruct(x0, PurificationConfig(T_START,K,linear_timesteps),noise)

In [ ]:
def _check_steps_inputs(K:int,t_start:int, schedule:DiffusionSchedule):
    if not 0 <= t_start < schedule.timesteps:
        raise ValueError("t_start must be in [0, schedule.timesteps - 1]")
    if not 1 <= K <= t_start + 1:
        raise ValueError("K must be in [1, t_start + 1]")

In [ ]:
lin_steps = linear_timesteps(K,T_START,schedule)

In [ ]:
def quadratic_timesteps(K:int, schedule:DiffusionSchedule, t_start:int,*, curve:float)->List[int]:
    _check_steps_inputs(K,t_start,schedule)
    n = K-1
    slope = (t_start- curve * n**2)/n
    steps = [round((curve*(x**2))+ slope*x) for x in range(K)] 
    return steps[::-1]

In [ ]:
log_snr_steps = uniform_log_snr_steps(K, T_START, schedule)
log_snr_steps

In [ ]:
x0_hat, scores = reverse_diffusion_ddim(xt,lin_steps,model,schedule)

In [ ]:
def residual_map(x0:torch.Tensor, x0_hat:torch.Tensor)-> torch.Tensor:
    return torch.abs(x0-x0_hat)/(1+torch.abs(x0_hat)) 

In [ ]:
residual_map(x0,x0_hat)

In [ ]:
def mean_residual_score(x0: torch.Tensor,x0_hat: torch.Tensor) -> torch.Tensor:
    e = torch.abs(x0 - x0_hat) / (1 + torch.abs(x0_hat))
    return e.flatten(start_dim=1).mean(dim=1)  # forme (B,)

In [ ]:
def score_config(model:UNetDiffusion, config:PurificationConfig,x0:torch.Tensor, eps:torch.Tensor, score_method:ScoreMethod):
    # Diffuse original data 
    t = torch.tensor([config.t_start], device=x0.device)
    xt, noise = diffuse(x0, t, config.schedule,noise=eps)
    # Generate purification steps
    steps = config.step_curve(config.K,config.t_start,config.schedule)
    # Purify 
    x0_hat, scores = reverse_diffusion_ddim(xt,steps,model,config.schedule)

    return score_method(x0,x0_hat)

In [ ]:
def normalize_patch(x0:torch.Tensor)->torch.Tensor:
    return (x0 - torch.mean(x0))/ torch.std(x0)

In [ ]:
import json
from pathlib import Path

dataset_dir = Path("../dataset")
test_dir = dataset_dir / "test"
annotations_path = dataset_dir / "annotations.json"

with annotations_path.open(encoding="utf-8") as file:
    annotations = json.load(file)

patches: dict[str, np.ndarray] = {}
for patch_name in sorted(annotations):
    patch_path = test_dir / patch_name
    if not patch_path.exists():
        raise FileNotFoundError(f"Patch annoté introuvable : {patch_path}")

    patch = np.load(patch_path).astype(np.float32).squeeze()
    if patch.shape != (48, 48):
        raise ValueError(f"Forme invalide pour {patch_name} : {patch.shape}")
    patches[patch_name] = patch

print(f"{len(patches)} patches annotés chargés")

In [ ]:
K_values = [2, 4, 6]

configs_K = [
    PurificationConfig(
        t_start=600,
        K=K,
        schedule=schedule,
        step_curve=linear_timesteps,
    )
    for K in K_values
]

In [ ]:
sweep_K = EvaluationSweep(
    model=model,
    configs=configs_K,
    patches=patches,
    annotations=annotations,
    score_method=mean_residual_score,
    device=device,
    batch_size= 32,
    seeds=[0, 1, 2],
)

results_K = sweep_K.run()

In [ ]:
results_K